In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder

# Cargar los datos de entrenamiento y prueba
train_df = pd.read_csv('E:/Bootcamp/titanic/train.csv')
test_df = pd.read_csv('E:/Bootcamp/titanic/test.csv')

# Mostrar las primeras filas para inspeccionar los datos
print(train_df.head())


   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  


In [2]:
# Imputar los valores faltantes en 'Age' con la mediana
train_df['Age'].fillna(train_df['Age'].median(), inplace=True)

# Imputar los valores faltantes en 'Embarked' con el valor más frecuente
train_df['Embarked'].fillna(train_df['Embarked'].mode()[0], inplace=True)

# Imputar los valores faltantes en 'Fare' (en caso de haber en los datos de test)
test_df['Fare'].fillna(test_df['Fare'].median(), inplace=True)

# Crear la variable binaria "CabinKnown" que indica si la cabina es conocida (1) o no (0)
train_df['CabinKnown'] = train_df['Cabin'].notna().astype(int)

# Mostrar si se imputaron correctamente los valores faltantes
print(train_df.isnull().sum())


PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age              0
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         0
CabinKnown       0
dtype: int64


C:\Users\chave\AppData\Local\Temp\ipykernel_20700\3350978899.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_df['Age'].fillna(train_df['Age'].median(), inplace=True)
C:\Users\chave\AppData\Local\Temp\ipykernel_20700\3350978899.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a 

In [3]:
# Tamaño de la familia: "FamilySize" combinando "SibSp" (hermanos/esposos) y "Parch" (padres/hijos)
train_df['FamilySize'] = train_df['SibSp'] + train_df['Parch'] + 1  # +1 para incluir al pasajero

# Crear la variable categórica "FareCategory" agrupando la tarifa en rangos
train_df['FareCategory'] = pd.cut(train_df['Fare'], bins=[0, 7.91, 14.454, 31, 513], 
labels=['Low', 'Medium', 'High', 'Very High'])

# Extraer el título de la variable "Name"
train_df['Title'] = train_df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

# Ver las nuevas características creadas
print(train_df[['FamilySize', 'FareCategory', 'Title', 'CabinKnown']].head())


   FamilySize FareCategory Title  CabinKnown
0           2          Low    Mr           0
1           2    Very High   Mrs           1
2           1       Medium  Miss           0
3           2    Very High   Mrs           1
4           1       Medium    Mr           0


<>:9: SyntaxWarning: invalid escape sequence '\.'
<>:9: SyntaxWarning: invalid escape sequence '\.'
C:\Users\chave\AppData\Local\Temp\ipykernel_20700\1101419574.py:9: SyntaxWarning: invalid escape sequence '\.'
  train_df['Title'] = train_df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)


In [4]:
# Convertir las variables categóricas a numéricas utilizando LabelEncoder
le = LabelEncoder()

# Aplicar LabelEncoder a las columnas categóricas
train_df['Sex'] = le.fit_transform(train_df['Sex'])
train_df['Embarked'] = le.fit_transform(train_df['Embarked'])
train_df['FareCategory'] = le.fit_transform(train_df['FareCategory'])
train_df['Title'] = le.fit_transform(train_df['Title'])

# Verificación de la transformación
print(train_df[['Sex', 'Embarked', 'FareCategory', 'Title']].head())


   Sex  Embarked  FareCategory  Title
0    1         2             1     12
1    0         0             3     13
2    0         2             2      9
3    0         2             3     13
4    1         2             2     12


In [5]:
# Seleccionar las características relevantes para el modelo
X = train_df[['Pclass', 'Sex', 'Age', 'Fare', 'FamilySize', 'CabinKnown', 'FareCategory', 'Title']]
y = train_df['Survived']

# Dividir el conjunto de datos en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Mostrar las dimensiones de los conjuntos de entrenamiento y prueba
print(f"Conjunto de entrenamiento: {X_train.shape}, Conjunto de prueba: {X_test.shape}")


Conjunto de entrenamiento: (712, 8), Conjunto de prueba: (179, 8)


In [6]:
# Instanciar el modelo
model = LogisticRegression(max_iter=1000)

# Entrenar el modelo con los datos de entrenamiento
model.fit(X_train, y_train)

# Hacer predicciones en los datos de prueba
y_pred = model.predict(X_test)


In [7]:
# Calcular la exactitud del modelo
accuracy = accuracy_score(y_test, y_pred)

# Calcular el F1-Score del modelo
f1 = f1_score(y_test, y_pred)

# Mostrar los resultados de la evaluación
print(f"Exactitud del modelo: {accuracy:.4f}")
print(f"F1-Score del modelo: {f1:.4f}")


Exactitud del modelo: 0.8212
F1-Score del modelo: 0.7746


In [8]:
# Imprimir los coeficientes del modelo para ver la importancia de las características
coef = pd.DataFrame(model.coef_[0], index=X.columns, columns=['Coeficiente'])
print(coef)


              Coeficiente
Pclass          -0.736791
Sex             -2.579675
Age             -0.030029
Fare             0.003837
FamilySize      -0.241580
CabinKnown       0.638816
FareCategory    -0.120588
Title           -0.052456
